In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, precision_recall_curve
)


In [ ]:
# ============================================================
# DATA IMPORT + TRAIN-TEST SPLIT (Run this first)
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# --------------------------------------------------
# 1. Load the dataset
# --------------------------------------------------
df = pd.read_excel("/content/default of credit card clients.xls")


print("Original shape:", df.shape)

# The first row contains the actual column headers. Let's set it as the header.
df.columns = df.iloc[0]
df = df[1:].copy()

# Rename the 'Unnamed: 0' column to 'ID' and the correctly named target column
df = df.rename(columns={'Unnamed: 0': 'ID', 'Y': 'default_payment_next_month'})

# Convert all relevant columns to numeric types
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Display info again to confirm data types and check for any remaining non-numeric issues
print("DataFrame after cleaning and type conversion:")
display(df.info())
display(df.head())

Original shape: (30001, 25)
DataFrame after cleaning and type conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 1 to 30000
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null

None

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
1,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
2,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
3,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
4,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
5,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [ ]:

# --------------------------------------------------
# 2. Basic cleaning
# --------------------------------------------------
if 'ID' in df.columns:
    df = df.drop(columns=['ID'])

# Check target column name (sometimes it is different)
print("\nColumns:", df.columns.tolist())

# Make sure the target column is named correctly
# Common names: 'default_payment_next_month' or 'default.payment.next.month'
target_col = 'default_payment_next_month'

if target_col not in df.columns:
    # try the other common name
    if 'default.payment.next.month' in df.columns:
        df = df.rename(columns={'default payment next month': target_col})
    else:
        raise ValueError("Target column not found. Please check the column name.")

# --------------------------------------------------
# 3. Separate Features (X) and Target (y)
# --------------------------------------------------
X = df.drop(columns=[target_col])
y = df[target_col]

print("\nFeatures shape:", X.shape)
print("Target distribution:")
print(y.value_counts(normalize=True).round(4) * 100)

# --------------------------------------------------
# 4. Train-Test Split (80% train / 20% test)
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # very important because of class imbalance
)

print("\nAfter split:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True).round(4) * 100)

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).round(4) * 100)


Columns: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']


ValueError: Target column not found. Please check the column name.

In [ ]:

# ============================================================
# 1. Scaling (Compulsory for Neural Networks)
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done.")
print("Training shape:", X_train_scaled.shape)

# ============================================================
# 2. Class weights (very important for imbalance)
# ============================================================
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

# ============================================================
# 3. Build the Artificial Neural Network
# ============================================================
model = Sequential([
    # Input + First Hidden Layer
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.35),

    # Second Hidden Layer
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.30),

    # Third Hidden Layer
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.25),

    # Fourth Hidden Layer (optional, can remove if overfitting)
    Dense(16, activation='relu'),
    Dropout(0.20),

    # Output Layer
    Dense(1, activation='sigmoid')
])

# ============================================================
# 4. Compile the model (all important parameters)
# ============================================================
model.compile(
    optimizer=Adam(learning_rate=0.001),      # good starting learning rate
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

model.summary()

# ============================================================
# 5. Callbacks (control training intelligently)
# ============================================================
early_stopping = EarlyStopping(
    monitor='val_auc',
    patience=20,                    # stop if no improvement for 20 epochs
    restore_best_weights=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_auc',
    factor=0.5,                     # reduce LR by half
    patience=8,
    min_lr=1e-6,
    mode='max',
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_ann_model.h5',
    monitor='val_auc',
    save_best_only=True,
    mode='max',
    verbose=0
)

# ============================================================
# 6. Train the model
# ============================================================
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,           # 20% of training data for validation
    epochs=150,                     # maximum epochs
    batch_size=256,                 # good for this dataset size
    class_weight=class_weight_dict,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1
)

# ============================================================
# 7. Predictions
# ============================================================
y_train_proba = model.predict(X_train_scaled, verbose=0).flatten()
y_test_proba  = model.predict(X_test_scaled, verbose=0).flatten()

y_train_pred = (y_train_proba >= 0.5).astype(int)
y_test_pred  = (y_test_proba >= 0.5).astype(int)

# ============================================================
# 8. Full Evaluation
# ============================================================
print("\n" + "="*60)
print("ARTIFICIAL NEURAL NETWORK - FINAL RESULTS")
print("="*60)

print("\n--- Accuracy ---")
print(f"Training Accuracy : {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Testing Accuracy  : {accuracy_score(y_test, y_test_pred):.4f}")

print("\n--- Precision / Recall / F1 (Class 1 = Default) ---")
print(f"Training Precision: {precision_score(y_train, y_train_pred):.4f}")
print(f"Testing Precision : {precision_score(y_test, y_test_pred):.4f}")
print(f"Training Recall   : {recall_score(y_train, y_train_pred):.4f}")
print(f"Testing Recall    : {recall_score(y_test, y_test_pred):.4f}")
print(f"Training F1       : {f1_score(y_train, y_train_pred):.4f}")
print(f"Testing F1        : {f1_score(y_test, y_test_pred):.4f}")

print("\n--- Ranking Metrics ---")
print(f"Training ROC-AUC  : {roc_auc_score(y_train, y_train_proba):.4f}")
print(f"Testing ROC-AUC   : {roc_auc_score(y_test, y_test_proba):.4f}")
print(f"Testing Avg Precision: {average_precision_score(y_test, y_test_proba):.4f}")

print("\n--- Classification Report (Test - threshold 0.5) ---")
print(classification_report(y_test, y_test_pred, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

# ============================================================
# 9. Find Best Threshold
# ============================================================
precisions, recalls, thresholds = precision_recall_curve(y_test, y_test_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]

print(f"\nBest threshold by F1: {best_threshold:.4f}")
print(f"Precision @ best: {precisions[best_idx]:.4f}")
print(f"Recall    @ best: {recalls[best_idx]:.4f}")
print(f"F1        @ best: {f1_scores[best_idx]:.4f}")

y_test_pred_opt = (y_test_proba >= best_threshold).astype(int)

print("\n--- Classification Report (Optimal Threshold) ---")
print(classification_report(y_test, y_test_pred_opt, digits=4))

# ============================================================
# 10. Plot Training History
# ============================================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['auc'], label='Train AUC')
plt.plot(history.history['val_auc'], label='Val AUC')
plt.title('AUC')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:

# ============================================================
# 1. Scaling (Compulsory for Neural Networks)
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done.")
print("Training shape:", X_train_scaled.shape)

# ============================================================
# 2. Class weights (very important for imbalance)
# ============================================================
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

# ============================================================
# 3. Build the Artificial Neural Network
# ============================================================
model = Sequential([
    # Input + First Hidden Layer
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.35),

    # Second Hidden Layer
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.30),

    # Third Hidden Layer
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.25),

    # Fourth Hidden Layer (optional, can remove if overfitting)
    Dense(16, activation='relu'),
    Dropout(0.20),

    # Output Layer
    Dense(1, activation='sigmoid')
])

# ============================================================
# 4. Compile the model (all important parameters)
# ============================================================
model.compile(
    optimizer=Adam(learning_rate=0.001),      # good starting learning rate
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

model.summary()

# ============================================================
# 5. Callbacks (control training intelligently)
# ============================================================
early_stopping = EarlyStopping(
    monitor='val_auc',
    patience=20,                    # stop if no improvement for 20 epochs
    restore_best_weights=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_auc',
    factor=0.5,                     # reduce LR by half
    patience=8,
    min_lr=1e-6,
    mode='max',
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_ann_model.h5',
    monitor='val_auc',
    save_best_only=True,
    mode='max',
    verbose=0
)

# ============================================================
# 6. Train the model
# ============================================================
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,           # 20% of training data for validation
    epochs=150,                     # maximum epochs
    batch_size=256,                 # good for this dataset size
    class_weight=class_weight_dict,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1
)

# ============================================================
# 7. Predictions
# ============================================================
y_train_proba = model.predict(X_train_scaled, verbose=0).flatten()
y_test_proba  = model.predict(X_test_scaled, verbose=0).flatten()

y_train_pred = (y_train_proba >= 0.5).astype(int)
y_test_pred  = (y_test_proba >= 0.5).astype(int)

# ============================================================
# 8. Full Evaluation
# ============================================================
print("\n" + "="*60)
print("ARTIFICIAL NEURAL NETWORK - FINAL RESULTS")
print("="*60)

print("\n--- Accuracy ---")
print(f"Training Accuracy : {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Testing Accuracy  : {accuracy_score(y_test, y_test_pred):.4f}")

print("\n--- Precision / Recall / F1 (Class 1 = Default) ---")
print(f"Training Precision: {precision_score(y_train, y_train_pred):.4f}")
print(f"Testing Precision : {precision_score(y_test, y_test_pred):.4f}")
print(f"Training Recall   : {recall_score(y_train, y_train_pred):.4f}")
print(f"Testing Recall    : {recall_score(y_test, y_test_pred):.4f}")
print(f"Training F1       : {f1_score(y_train, y_train_pred):.4f}")
print(f"Testing F1        : {f1_score(y_test, y_test_pred):.4f}")

print("\n--- Ranking Metrics ---")
print(f"Training ROC-AUC  : {roc_auc_score(y_train, y_train_proba):.4f}")
print(f"Testing ROC-AUC   : {roc_auc_score(y_test, y_test_proba):.4f}")
print(f"Testing Avg Precision: {average_precision_score(y_test, y_test_proba):.4f}")

print("\n--- Classification Report (Test - threshold 0.5) ---")
print(classification_report(y_test, y_test_pred, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

# ============================================================
# 9. Find Best Threshold
# ============================================================
precisions, recalls, thresholds = precision_recall_curve(y_test, y_test_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]

print(f"\nBest threshold by F1: {best_threshold:.4f}")
print(f"Precision @ best: {precisions[best_idx]:.4f}")
print(f"Recall    @ best: {recalls[best_idx]:.4f}")
print(f"F1        @ best: {f1_scores[best_idx]:.4f}")

y_test_pred_opt = (y_test_proba >= best_threshold).astype(int)

print("\n--- Classification Report (Optimal Threshold) ---")
print(classification_report(y_test, y_test_pred_opt, digits=4))

# ============================================================
# 10. Plot Training History
# ============================================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['auc'], label='Train AUC')
plt.plot(history.history['val_auc'], label='Val AUC')
plt.title('AUC')
plt.legend()

plt.tight_layout()
plt.show()